### fact_bookings


In [0]:
use catalog identifier(:catalog)

In [0]:
create or replace table gold.fact_booking as
select 
fb.fb_bookingid as booking_id, 
fb.fb_routeid as route_id, 
fb.fb_airlineid as airline_id, 
fb.fb_linenumber as line_number, 
fb.fb_seats as seats, 
fb.fb_ticketprice as ticket_price, 
fb.fb_discount as discount, 
fb.fb_tax as tax, 
fb.fb_refundflag as refund_flag, 
fb.fb_segmentstatus as srgment_status,
fb.fb_commitdate as commit_date, 
fb.fb_departuredate as departure_date, 
fb.fb_arrivaldate as arrival_date, 
fb.fb_cabinclass as cabin_class, 
b.bk_passengerid as passenger_id, 
b.bk_status as booking_status, 
b.bk_totalprice as total_price, 
b.bk_bookingdate as booking_date, 
b.bk_flightdate as flight_date, 
b.bk_bookingpriority as booking_priority, 
b.bk_agent as agent, 
p.pa_nationkey as nationkey
from silver.flight_bookings fb
join silver.bookings b
on fb.fb_bookingid = b.bk_bookingid
join silver.passengers p
on b.bk_passengerid = p.pa_passengerid

### dim passenger

In [0]:
create or replace table gold.dim_passenger as
select * except(pa_comment) from silver.passengers

### dim route

In [0]:
create or replace table gold.dim_rooute as
select * except(ro_comment) from silver.routes

### dim airline

In [0]:
create or replace table gold.dim_airline as
select * except(al_comment) from silver.airlines

### dim geography

In [0]:
create or replace table gold.dim_geography as
select n.n_nationkey, n.n_name, n.n_regionkey, r.r_name 
from silver.nations n
join silver.regions r
on n.n_regionkey = r.r_regionkey

### Annual booking kpis

In [0]:
-- annual net revenue, distinct bookings, active passengers, average booking value 
create view if not exists vw_annual_booking_kpis as
select year(booking_date) as year,
round(sum(ticket_price * (1 - discount)),2) as net_revenue,
count(distinct booking_id) as distinct_bookings,
count(distinct passenger_id) as active_passengers,
round(net_revenue / distinct_bookings,2) as average_booking_value
from gold.fact_booking
group by year(booking_date)